# Phase 1 Supervised Fine-Tuning - Qwen 2.5-0.5B on Colab T4

End-to-end SFT run for the Phase 1 baseline: train `Qwen/Qwen2.5-0.5B` on GSM8K + MATH using the production `Trainer` (the same one exercised locally with tiny-gpt2).

**Two paths in this notebook:**

1. **Primary path - Qwen 2.5-0.5B via the production CLI.** One shell command, one YAML config (`experiments/colab_qwen_500m.yaml`). Trainer handles seeding, packing, masked cross-entropy, warmup+cosine learning-rate schedule, gradient accumulation, gradient clipping, periodic validation, periodic checkpointing, and Weights & Biases logging. This is the path you will use for every real Phase 1 run.

2. **Optional path - Gemma 3 1B with memory tricks.** A separate cell at the bottom that imports `Trainer` directly, enables gradient checkpointing, and swaps in 8-bit AdamW from `bitsandbytes`. Together these three knobs (gradient checkpointing, 8-bit optimiser, batch=1 with grad-accum=4) are what make a 1B-class model fit on a free T4. **Skippable** - and **gated** - see the Gemma section.

**Before running:** set the runtime to T4 GPU. *Runtime -> Change runtime type -> T4 GPU.* On CPU this notebook will run, but a single Qwen step takes minutes, not seconds.

---


## Step 1 - verify the GPU

If `nvidia-smi` errors or shows no GPU, the runtime is not set to T4. Stop here and switch it. Training Qwen 0.5B on a CPU runtime would take hours per epoch; on T4 it takes ~3 seconds per step at the conservative settings in `experiments/colab_qwen_500m.yaml`.


In [ ]:
!nvidia-smi

## Step 2 - clone the repo and install dependencies

We clone the public finpost repo and install it in editable mode (`pip install -e .`). Editable mode means we install the *source tree* itself as the package - so `from finpost.training.trainer import Trainer` works exactly as it does on your local machine. No copy-pasting code into cells.

The `transformers`, `torch`, `datasets`, `pyyaml`, `pydantic`, `wandb`, `safetensors` deps are listed in `pyproject.toml` and pip will resolve them. Colab usually has compatible versions of torch and transformers preinstalled, so this finishes in 1-2 minutes.


In [ ]:
!git clone https://github.com/shannan-liu1/finpost.git
%cd finpost
!pip install -q -e .

## Step 3 - sys.path workaround for editable installs on Colab Python 3.12

`pip install -e .` is supposed to make `finpost` importable. On Colab's Python 3.12 the hatchling editable build sometimes registers the package such that direct imports work but submodule discovery (`finpost.training.trainer`) does not. The fix that always works is to put the source root on `sys.path` ourselves. This is a known issue, not something specific to this repo - leaving it in even after a future Colab Python upgrade does no harm.


In [ ]:
import sys

# /content/finpost is where the clone above lands on Colab; src/ is the
# package root inside the repo. Insert at index 0 so this entry wins over
# any conflicting one Colab might have.
sys.path.insert(0, '/content/finpost/src')

# Sanity check: the import below must succeed before continuing.
import finpost
import finpost.training.trainer  # noqa: F401  - proves submodule resolution works
print('finpost importable from:', finpost.__file__)

## Step 4 - confirm CUDA visibility from PyTorch

`nvidia-smi` shows what the host sees; `torch.cuda.is_available()` shows what the Python process sees. They should agree. If not, the runtime is misconfigured.


In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:        ', torch.cuda.get_device_name(0))
    print('VRAM (GB):     ', torch.cuda.get_device_properties(0).total_memory / 1e9)
else:
    print('No CUDA. Switch the runtime to T4 GPU and re-run from the top.')

## Step 5 - primary training run via the production CLI

This is the cell that does the work. `python -m finpost.training.train` is the production entry point: parses the YAML, validates the config through Pydantic, constructs `Trainer(config)`, and runs `.train()`.

The config we pass is `experiments/colab_qwen_500m.yaml` - a deliberately conservative 500-step run with `batch_size=4`, `grad_accum_steps=4`, `max_seq_len=1024`. On T4 that's ~25 minutes wall-clock. Inline comments in the YAML explain every value.

**Weights & Biases:**
- If you have a `WANDB_API_KEY` set (Colab Secrets -> name `WANDB_API_KEY` -> toggle "notebook access"), the run uploads live and you can plot from `wandb.Api()` in the next cell.
- If not, set `WANDB_MODE=offline` first (uncomment the line in the cell). Plots from the offline run require `wandb sync wandb/offline-run-*` after training, then the same `wandb.Api()` call. The dashboard URL Trainer prints at startup is the canonical place to see curves either way.


In [ ]:
# Uncomment the next line if you do NOT have a WANDB_API_KEY in Colab Secrets.
# %env WANDB_MODE=offline

# Authenticated path: pull the API key from Colab Secrets if present.
# google.colab.userdata is only available inside Colab; the try/except
# makes the cell a no-op when run elsewhere (e.g. local Jupyter).
try:
    from google.colab import userdata  # type: ignore
    import os
    key = userdata.get('WANDB_API_KEY')
    if key:
        os.environ['WANDB_API_KEY'] = key
        print('WANDB_API_KEY loaded from Colab Secrets.')
    else:
        print('No WANDB_API_KEY in Colab Secrets - wandb will run anonymously or offline.')
except Exception:
    pass

# Single command: YAML in, trained-and-checkpointed model out.
# --device cuda forces GPU even though Trainer's default is "cuda if
# available", because we want a loud failure (rather than a silent
# fallback) if CUDA is gone.
!python -m finpost.training.train --config experiments/colab_qwen_500m.yaml --device cuda

## Step 6 - plot loss and tokens-per-second from Weights & Biases

We don't reimplement plotting on top of wandb's binary log format. The path that always works is `wandb.Api()` - it returns the run history as a pandas DataFrame, and matplotlib plots it directly.

If you ran offline above, run `!wandb sync wandb/offline-run-*` once before this cell so the run shows up on the wandb server. If you have no wandb account at all, skip this cell - the curves still exist in the binary `.wandb` file inside `wandb/offline-run-*/`, just not in a format easy to plot in-notebook.


In [ ]:
import wandb
import matplotlib.pyplot as plt

# Project + run_name match experiments/colab_qwen_500m.yaml.
PROJECT = 'finpost-phase1'
RUN_NAME = 'qwen2.5-0.5b-colab-demo'

api = wandb.Api()
# Filter on display_name; if your default entity isn't set, prepend it as
# f'{entity}/{PROJECT}'. Most Colab users have one default entity so the
# bare project name resolves fine.
runs = api.runs(PROJECT, filters={'display_name': RUN_NAME})
if not runs:
    raise RuntimeError(
        f'No wandb run named {RUN_NAME!r} in project {PROJECT!r}. '
        'If you ran offline, run `!wandb sync wandb/offline-run-*` first.'
    )
run = runs[0]
# history() returns a pandas DataFrame, one row per logged step.
history = run.history(keys=['train/loss', 'train/tokens_per_sec'], pandas=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history['_step'], history['train/loss'], linewidth=1)
axes[0].set_xlabel('step')
axes[0].set_ylabel('train/loss')
axes[0].set_title('Training loss')
axes[0].grid(alpha=0.3)

# tokens_per_sec is logged every 10 steps (per Trainer's amendment), so
# the series will have NaN gaps. Drop them before plotting.
tps = history.dropna(subset=['train/tokens_per_sec'])
axes[1].plot(tps['_step'], tps['train/tokens_per_sec'], marker='o', markersize=3, linewidth=1)
axes[1].set_xlabel('step')
axes[1].set_ylabel('tokens/sec')
axes[1].set_title('Throughput (response tokens/sec)')
axes[1].grid(alpha=0.3)

fig.suptitle(f'{RUN_NAME} - {len(history)} logged steps', y=1.02)
plt.tight_layout()
plt.show()

## Optional - Gemma 3 1B on T4 with memory tricks

Skip this section unless you want to push to a 1B-class model on the same hardware. The Qwen 500m run above is the headline path.

**Why this is a separate, more invasive cell:** Trainer is configured via a Pydantic schema that doesn't expose `gradient_checkpointing` or `bitsandbytes` fields - adding them is a real Trainer enhancement, out of scope for this notebook. Instead, we instantiate `Trainer(config)`, call its private `_setup()` to load the model and build the optimiser, then PATCH the optimiser and enable gradient checkpointing before the loop runs. This is honest about the trade-off: we're poking at internals deliberately, not pretending the schema supports it.

**Three memory tricks, each does one specific thing:**

1. **Gradient checkpointing** (`model.gradient_checkpointing_enable()`). Standard activation-recomputation. Activations from the forward pass are dropped; backward recomputes them. Trades roughly 30% extra compute for a roughly 60% drop in activation memory. The exact numbers vary by model architecture, but the trend is reliable across modern transformers.
2. **8-bit AdamW** (`bitsandbytes.optim.AdamW8bit`). Stores the two AdamW momentum buffers in 8-bit instead of 32-bit. For a 1B model this cuts optimiser state from ~8 GB to ~1 GB. Block-wise quantisation keeps numerical fidelity; the 8-bit AdamW paper reports parity with 32-bit on most SFT runs.
3. **Gradient accumulation** (`per_device_batch_size=1, grad_accum_steps=4`). Keep activations small per microbatch; combine four microbatches into one effective batch of 4. This is already in the YAML - Trainer applies it natively.

Together these let `google/gemma-3-1b-it` (about 1.0 B parameters) fit on a 15 GB T4 alongside its activations, gradients, and optimiser state. Without them, OOM on backward.

**Requirements before running:**
- `pip install bitsandbytes` (the `bnb` Linux wheel; not available on macOS).
- Hugging Face access to `google/gemma-3-1b-it` - Gemma models are gated. Accept the licence on the model page.
- A Hugging Face token with read access to that gated repo, stored in Colab Secrets as `FINPOST_LOCAL_GEMMA`. The cell logs in with it; without it, model download fails with a 401.


In [ ]:
# Install bitsandbytes if needed. Quiet flag keeps Colab tidy.
!pip install -q bitsandbytes

In [ ]:
import os
from pathlib import Path

import torch
import bitsandbytes as bnb
from huggingface_hub import login as hf_login

from finpost.training.config import Config
from finpost.training.trainer import Trainer

# 1. Hugging Face authentication - Gemma is gated.
try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get('FINPOST_LOCAL_GEMMA')
    if not hf_token:
        raise RuntimeError(
            'No FINPOST_LOCAL_GEMMA secret found. Add a Colab Secret named '
            '`FINPOST_LOCAL_GEMMA` containing a HF token with read access '
            'to google/gemma-3-1b-it.'
        )
    hf_login(token=hf_token)
    print('HF login successful.')
except ImportError:
    # Not on Colab - assume `huggingface-cli login` was run.
    pass

# 2. Build the Config inline rather than via YAML - Gemma's identifier
# and the smaller batch size are intentional one-offs for this section
# and don't need to live in experiments/. Everything else mirrors the
# Qwen demo so the pedagogical comparison is apples-to-apples.
gemma_config = Config.model_validate({
    'model': {
        'base_model_id': 'google/gemma-3-1b-it',
        'dtype': 'bfloat16',
        'use_safetensors': True,
    },
    'data': {
        'sources': ['gsm8k', 'math'],
        'val_split_pct': 5.0,
        'seed': 42,
    },
    'training': {
        # Same optimiser-step budget as the Qwen demo so the loss
        # curves are comparable. Memory tricks make this fit; without
        # them, OOM at step 0.
        'max_steps': 500,
        'warmup_steps': 50,
        'lr': 2.0e-5,
        'weight_decay': 0.01,
        # Per-device 1, accumulate 4. Effective batch = 4 - half the
        # Qwen demo's 16, but Gemma is twice the parameter count, so
        # tokens-per-step is in the same ballpark.
        'per_device_batch_size': 1,
        'grad_accum_steps': 4,
        'grad_clip': 1.0,
        'val_every_n_steps': 100,
        'checkpoint_every_n_steps': 250,
    },
    'packing': {
        'max_seq_len': 1024,
        'isolate_documents': True,
    },
    'logging': {
        'wandb_project': 'finpost-phase1',
        'run_name': 'gemma3-1b-colab-demo',
    },
    'checkpointing': {
        'save_dir': 'results/checkpoints/gemma3-1b-colab-demo',
        'retention_last_n': 2,
    },
})

# 3. Construct Trainer. _setup() loads the model + tokenizer + loaders +
# optimiser + scheduler - all the expensive work. We then OVERWRITE the
# optimiser and enable gradient checkpointing on the model BEFORE the
# loop reads either.
trainer = Trainer(gemma_config)
trainer._setup()
assert trainer.model is not None  # narrows the type for linters
assert trainer.optimizer is not None

# Gradient checkpointing - call AFTER .to(device) (Trainer did this in
# _setup) and BEFORE the training loop. Internally this wraps each
# transformer block with `torch.utils.checkpoint.checkpoint`.
trainer.model.gradient_checkpointing_enable()
print('Gradient checkpointing enabled on', type(trainer.model).__name__)

# Swap the optimiser. AdamW8bit takes the same args as torch.optim.AdamW.
# Why we rebuild from scratch instead of reusing param groups: the
# original optimiser is the standard fp32 AdamW Trainer built; replacing
# the whole instance is cleaner than mutating its internal state. The
# scheduler is keyed on the original optimiser object - we have to
# re-bind it to the new one or the lr update on each .step() will be
# applied to the wrong optimiser.
trainer.optimizer = bnb.optim.AdamW8bit(
    trainer.model.parameters(),
    lr=gemma_config.training.lr,
    weight_decay=gemma_config.training.weight_decay,
)
# Re-bind the scheduler. LRScheduler's optimizer attribute is just a
# Python reference - overwriting it is supported.
assert trainer.scheduler is not None
trainer.scheduler.optimizer = trainer.optimizer
print('Optimiser swapped to', type(trainer.optimizer).__name__)

# 4. Run the loop and tear down. We deliberately do NOT call
# trainer.train() - that calls _setup() again, which would discard our
# patches.
trainer._run_training_loop()
trainer._teardown()
print('Gemma 1B run complete. Final step:', trainer.global_step)

## Next steps

If the Qwen run looks healthy:

1. **Longer run** - copy `experiments/colab_qwen_500m.yaml`, raise `max_steps` to 3000 (the published `experiments/baseline.yaml` value), and re-run. Expect ~2-3 hours on T4.
2. **The multi-task notebook** - open `notebooks/sft_phase1_multitask.ipynb`. It runs three Qwen 0.5B arms (combined / GSM8K-only / MATH-only) and compares them on both test sets. That's the headline transfer-learning experiment for Phase 1.
3. **Resume training** - pass `--resume-from results/checkpoints/qwen2.5-0.5b-colab-demo/step-XXXXXXXX/` to continue from a checkpoint. RNG state, optimiser state, scheduler state, and step counter all round-trip; the DataLoader iterator does not (a known limitation, see `Trainer`'s docstring).
4. **Eval harness** - generation-based accuracy on GSM8K + MATH test sets is its own workstream. The eval cell in the multi-task notebook is loss-only on purpose; passing rate against the verifier comes later.

If the run looks unhealthy (loss spikes, NaN, flat loss after 100 steps): drop `lr` to 5e-6 or raise `warmup_steps`, switch `dtype` to `float32` to rule out bf16 instability, and re-run with `--max-steps 50` to keep the diagnostic loop fast.
